# Monthly net sales and bill count trend, with readable month names


In [0]:
q1 = spark.sql("""
SELECT
  s.bill_month_start,
  d.bill_year,
  d.bill_month_name,
  SUM(s.total_net_sales) AS total_net_sales,
  SUM(s.total_bills)     AS total_bills
FROM gold_vw_sales_summary_monthly s
JOIN (
  SELECT DISTINCT bill_month_start, bill_year, bill_month_name
  FROM gold_vw_dim_date
) d
  ON s.bill_month_start = d.bill_month_start
GROUP BY s.bill_month_start, d.bill_year, d.bill_month_name
ORDER BY s.bill_month_start
""")
display(q1)

bill_month_start,bill_year,bill_month_name,total_net_sales,total_bills
2024-01-01,2024,January,2.1666902530157775E7,71376
2024-02-01,2024,February,2.0575534976510264E7,67902
2024-03-01,2024,March,2.215209211021822E7,72532
2024-04-01,2024,April,2.376231037399146E7,75340
2024-05-01,2024,May,2.5173129261982568E7,78592
2024-06-01,2024,June,2.647229132769078E7,82586
2024-07-01,2024,July,3.163409762027216E7,94996
2024-08-01,2024,August,3.3379777617738422E7,99190
2024-09-01,2024,September,3.258473935027767E7,98540
2024-10-01,2024,October,3.460635649980417E7,102040


# Total net sales and bills per store

In [0]:
q2 = spark.sql("""
SELECT
  store_id,
  SUM(total_net_sales) AS total_net_sales,
  SUM(total_bills)     AS total_bills
FROM gold_vw_sales_summary_monthly
GROUP BY store_id
ORDER BY total_net_sales DESC
""")
display(q2)

store_id,total_net_sales,total_bills
17267,7.559719981997429E7,176805
17681,6.813209618983087E7,201877
18378,6.6311439439719155E7,139035
18955,6.58680045980772E7,123827
18691,6.234395295933424E7,146791
19096,6.0245122724581465E7,167981
18529,5.811798080570473E7,123783
17750,5.51436854636762E7,146242
25243,5.1392150255991444E7,118792
19095,5.021685895215759E7,124958


# Revenue % contribution by category 

In [0]:
q3 = spark.sql("""
SELECT
  category,
  SUM(net_sales_value) AS net_sales,
  ROUND(SUM(net_sales_value) * 100.0 / SUM(SUM(net_sales_value)) OVER (), 2) AS pct_of_total
FROM gold_vw_fact_sales_daily
GROUP BY category
ORDER BY net_sales DESC
""")
display(q3)

category,net_sales,pct_of_total
PHARMA,7.370270005635493E8,53.66
FMCG,4.633081862877957E8,33.73
PRIVATE LABEL,1.3396519996941453E8,9.75
SURGICAL,1.7727381072012067E7,1.29
CIRCLE SUBSCRIPTION,1.4204737E7,1.03
AYURVEDIC,7389100.949244022,0.54
STATIONERY,390.0,0.0


# Top 20 items by net sales, enriched with pack size and generic flag 

In [0]:
q4_top = spark.sql("""
SELECT
  t.item_id, t.item_name, t.category, t.total_units_sold, t.total_net_sales,
  i.pack_size, i.generic_flag
FROM gold_vw_top_selling_items t
LEFT JOIN gold_vw_dim_item i ON t.item_id = i.item_id
ORDER BY t.sales_rank
LIMIT 20
""")
display(q4_top)

item_id,item_name,category,total_units_sold,total_net_sales,pack_size,generic_flag
CIR0033,CIRCLE MEMBERSHIP,CIRCLE SUBSCRIPTION,97969,1.4220331E7,1,null
MAM0109,MAMYPOKO EXTRA ABSORB DIAPER PANTS XL 46'S,FMCG,5269,5787277.386474609,1,null
PAM0269,PAMPERS COMPLETE SKIN COMFORT PANTS XL 56'S,FMCG,3639,5189712.310058594,1,null
NAN0007,NAN PRO STAGE-2 FOLLOW-UP FORMULA POWDER 400G REFILL,FMCG,5630,4544643.772399902,1,null
APD0073,AP DISINFECTANT FLOOR CLEANER 400ML,PRIVATE LABEL,84295,4221500.0,1,null
NAN0070,NAN PRO STAGE-4 FOLLOW-UP FORMULA POWDER 400G REFILL,FMCG,4762,3819570.0326538086,1,null
APR0111,AP REFRESHING WIPES CITRUS 30'S,PRIVATE LABEL,44407,3813540.0,1,null
URI0004,URIMAX D TAB 15'S,PHARMA,86133,3534943.2535476685,15,null
NAN0008,NAN PRO STAGE-1 INFANT FORMULA POWDER 400G REFILL,FMCG,4303,3469171.7041015625,1,null
APB0018,"APOLLO PHARMACY BLOOD GLUCOSE TEST STRIPS, 50 COUNT",PRIVATE LABEL,4397,3438979.294128418,1,null


# Bottom 20 items by net sales, i.e. slow-mover candidates

In [0]:
q4_slow_movers = spark.sql("""
SELECT
  t.item_id, t.item_name, t.category, t.total_units_sold, t.total_net_sales,
  i.pack_size, i.generic_flag
FROM gold_vw_top_selling_items t
LEFT JOIN gold_vw_dim_item i ON t.item_id = i.item_id
ORDER BY t.sales_rank DESC
LIMIT 20
""")
display(q4_slow_movers)

item_id,item_name,category,total_units_sold,total_net_sales,pack_size,generic_flag
FRE0272,FREE COST - BP MONITOR,PRIVATE LABEL,1,0.009999999776482582,1,null
APD0066,AP DEODORANT FRAGRANCE BODY SPRAY 80ML (TESTER),PRIVATE LABEL,3,0.029999999329447746,1,null
NAN0149,NAN GROW FIRST FUN ACTIVITY BOOK,FMCG,51,0.509999992325902,1,null
VIC0017,VICKS COUGH DROPS 125`S,FMCG,1,1.0,125,null
BAN0039,BAND AID WASHPROOF,FMCG,1,2.5,130,null
VIC0537,VICKS DOUBLE POWER COUGH DROPS 2.7G*115'S (25 FREE),FMCG,2,4.0,115,null
TRO0032,TROPINE 0.6MG INJ 1ML,PHARMA,1,4.53000020980835,1,null
APU0003,AP Umbrella Free (Monsoon Offer),PRIVATE LABEL,550,5.49999987706542,1,null
BET0587,BETLAN 8MG TAB 10'S,PHARMA,1,5.5,10,null
COF0043,COFSILS GINGER LEMON LOZENGES 220'S,FMCG,2,6.0,220,GENERIC


# Distinct generic_flag values within PHARMA category

In [0]:
distinct_generic_flags = spark.sql("""
SELECT generic_flag, COUNT(*) AS n
FROM silver_vw_sales
WHERE category = 'PHARMA'
GROUP BY generic_flag
""")
display(distinct_generic_flags)

generic_flag,n
null,3171056
GENERIC,421733


# Monthly PHARMA sales split by generic vs branded

In [0]:
q5 = spark.sql("""
SELECT
  bill_month_start,
  COALESCE(generic_flag, 'UNKNOWN') AS generic_flag,
  SUM(sale_value) AS net_sales
FROM silver_vw_sales
WHERE category = 'PHARMA'
GROUP BY bill_month_start, generic_flag
ORDER BY bill_month_start
""")
display(q5)

bill_month_start,generic_flag,net_sales
2024-01-01,GENERIC,581562.5532758236
2024-01-01,UNKNOWN,1.1957754625724792E7
2024-02-01,UNKNOWN,1.128356815569067E7
2024-02-01,GENERIC,521827.47995114326
2024-03-01,UNKNOWN,1.1943129400880337E7
2024-03-01,GENERIC,533676.437066555
2024-04-01,UNKNOWN,1.2758403218225837E7
2024-04-01,GENERIC,551214.1846904755
2024-05-01,UNKNOWN,1.3503104264582872E7
2024-05-01,GENERIC,573014.33624053


# Discount % and gross/net sales by category and sub-category 

In [0]:
q6 = spark.sql("""
SELECT
  category,
  sub_category,
  discount_pct,
  gross_sales,
  net_sales
FROM gold_vw_discount_analysis
ORDER BY discount_pct DESC
""")
display(q6)

category,sub_category,discount_pct,gross_sales,net_sales
PRIVATE LABEL,GUMMIES,50.0,13347.54,6673.790234373882
FMCG,BISCUITS,34.15,77450.59999999999,51001.488273620605
FMCG,AIR FRESHNER,33.66,938892.26,622890.0
FMCG,DRY FRUITS,33.35,1379468.6799999997,919442.6899108887
FMCG,LINERS,33.33,8100.0,5400.0
FMCG,SOLUTION,32.95,120707.08,80940.0
FMCG,OATS,32.66,20270.5,13650.0
FMCG,OIL,32.3,4150590.3499999978,2809908.9286727905
FMCG,SPRAY,32.16,1.8957170180000126E7,1.2861473588523865E7
FMCG,BODY WASH,31.71,1016303.5800000001,694019.0


# Return rate by store and category (combos under 100 lines excluded)

In [0]:
q7 = spark.sql("""
SELECT
  store_id,
  category,
  COUNT(*)                                     AS total_lines,
  SUM(is_return)                                AS return_lines,
  ROUND(SUM(is_return) * 100.0 / COUNT(*), 2)   AS return_rate_pct
FROM silver_vw_sales
GROUP BY store_id, category
HAVING COUNT(*) > 100
ORDER BY return_rate_pct DESC
""")
display(q7)

store_id,category,total_lines,return_lines,return_rate_pct
18529,SURGICAL,6536,443,6.78
17772,SURGICAL,197,7,3.55
16257,SURGICAL,11072,362,3.27
18529,PHARMA,126362,3035,2.40
14783,SURGICAL,301,7,2.33
17267,SURGICAL,7597,160,2.11
19076,SURGICAL,2329,45,1.93
14783,AYURVEDIC,105,2,1.90
18529,AYURVEDIC,670,12,1.79
18206,PHARMA,4556,77,1.69


# Average bill value per store 

In [0]:
q8_by_store = spark.sql("""
SELECT store_id, ROUND(AVG(avg_bill_value), 2) AS avg_bill_value
FROM gold_vw_sales_summary_monthly
GROUP BY store_id
ORDER BY avg_bill_value DESC
""")
display(q8_by_store)

store_id,avg_bill_value
18955,525.75
25870,513.68
25249,508.7
18693,497.87
19076,478.17
18378,475.13
18896,469.54
18529,467.73
25243,429.52
18692,427.71


# Average bill value per customer group


In [0]:
q8_by_cust_group = spark.sql("""
SELECT
  cust_group_id,
  COUNT(DISTINCT bill_no)                               AS total_bills,
  ROUND(SUM(sale_value) / COUNT(DISTINCT bill_no), 2)   AS avg_bill_value
FROM silver_vw_sales
GROUP BY cust_group_id
ORDER BY avg_bill_value DESC
""")
display(q8_by_cust_group)

cust_group_id,total_bills,avg_bill_value
5004,1,4439.9
1432,2044,4186.46
8910,26,3152.57
9970,607,2943.2
2869,1,2400.0
3255,1,1700.1
8999,115,1383.56
5427,870,1333.25
4091,1,1299.0
2675,1,1244.0
